# Milestone 2: Data Cleaning dan Quality Profiling — Trase Indonesia Palm Oil

Notebook ini disesuaikan untuk dataset **Indonesia Palm Oil Supply Chain** dari Trase, file `indonesia_palm_oil_v1_2_5.csv`.

**Target Milestone 2:**
1. Membaca dataset CSV berukuran besar dengan Polars.
2. Menjalankan profiling DuckDB `SUMMARIZE`.
3. Memeriksa missing values dan nilai kosong.
4. Menangani missing values secara konservatif.
5. Mendeteksi outlier pada kolom kuantitatif yang relevan, tanpa menganggap tahun, kode, ID, skor, atau koordinat sebagai nilai outlier umum.
6. Menghapus baris duplikat identik, karena satu kolom kunci unik belum dikonfirmasi untuk data rantai pasok ini.
7. Menjalankan validasi data dasar dan menyimpan hasil ke `data/processed/`.

> Jalankan notebook dari root repository `tugas1-abd-2026`. File CSV asli harus berada di `data/raw/indonesia_palm_oil_v1_2_5.csv`. Notebook ini tidak mengasumsikan nama kolom bisnis yang belum diverifikasi; kolom akan diperiksa saat notebook dijalankan.


In [ ]:
from pathlib import Path
import re
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indonesia_palm_oil_v1_2_5.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "indonesia_palm_oil_bersih.parquet"
REPORT_PATH = PROJECT_ROOT / "data" / "processed" / "ringkasan_cleaning.txt"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}\n"
        "Pastikan file CSV Trase sudah diletakkan di folder data/raw/."
    )

print(f"File input : {DATA_PATH}")
print(f"Ukuran file: {DATA_PATH.stat().st_size / (1024**2):,.2f} MB")

# Baca CSV menggunakan Polars. infer_schema_length yang cukup besar membantu
# Polars mengenali tipe data pada dataset besar tanpa membaca seluruh file untuk inferensi.
df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
    ignore_errors=False,
    low_memory=True,
    null_values=["", "NA", "N/A", "NULL", "null", "None"],
)

print(f"Dimensi awal: {df.height:,} baris × {df.width} kolom")
print(f"Penggunaan memori perkiraan DataFrame: {df.estimated_size('mb'):,.2f} MB")
print("Nama kolom:")
print(df.columns)
df.head()


## 1. Profiling dengan DuckDB `SUMMARIZE`

`SUMMARIZE` menghasilkan ringkasan tipe data dan statistik dasar per kolom. Hasil ini menjadi baseline sebelum pembersihan.


In [ ]:
con = duckdb.connect()
con.register("dataset_awal", df)

profil_awal = con.execute("SUMMARIZE SELECT * FROM dataset_awal").pl()
profil_awal


## 2. Deteksi Missing Values

Selain nilai `NULL`, kolom teks dapat berisi string kosong atau spasi. Bagian ini menormalkan string kosong menjadi null, lalu menghitung jumlah dan persentase null per kolom.


In [ ]:
# Normalisasi string kosong/spasi menjadi null untuk kolom bertipe teks.
kolom_teks_awal = [
    name for name, dtype in zip(df.columns, df.dtypes)
    if dtype == pl.String
]
df = df.with_columns([
    pl.when(pl.col(c).str.strip_chars() == "")
      .then(None)
      .otherwise(pl.col(c))
      .alias(c)
    for c in kolom_teks_awal
])

def ringkasan_null(data: pl.DataFrame) -> pl.DataFrame:
    n = data.height
    if n == 0:
        return pl.DataFrame({"kolom": [], "jumlah_null": [], "persen_null": []})
    return (
        data.null_count()
        .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
        .with_columns(
            (pl.col("jumlah_null") / n * 100).round(2).alias("persen_null")
        )
        .filter(pl.col("jumlah_null") > 0)
        .sort("jumlah_null", descending=True)
    )

null_summary_awal = ringkasan_null(df)
print(f"Total nilai null: {df.null_count().sum_horizontal().sum():,}")
null_summary_awal


## 3. Penanganan Missing Values

Strategi yang digunakan:
- Kolom numerik kontinu yang memiliki null diisi dengan **median kolom** agar tidak terlalu sensitif terhadap nilai ekstrem.
- Kolom teks yang memiliki null diisi dengan label `Tidak diketahui`, sehingga baris rantai pasok tidak langsung dibuang.
- Kolom numerik yang tampak sebagai tahun, ID, kode, skor, atau koordinat **tidak otomatis diisi median**; null dipertahankan agar tidak membuat nilai identitas/lokasi sintetis.
- Kolom numerik dengan nama yang menunjukkan ukuran/volume/kuantitas/produksi/emisi dapat diisi median jika memiliki nilai yang tersedia.

Periksa daftar kolom yang dipilih sebelum menerapkan perubahan. Data sumber tetap tidak ditimpa.


In [ ]:
# Deteksi kolom numerik yang berpotensi berupa ukuran kontinu berdasarkan nama.
numeric_types = (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16,
                 pl.UInt32, pl.UInt64, pl.Float32, pl.Float64)
numeric_cols = [
    name for name, dtype in zip(df.columns, df.dtypes)
    if dtype in numeric_types
]

def nama_mirip_identitas_atau_kategori(name: str) -> bool:
    s = name.lower()
    tokens = ["id", "code", "kode", "year", "tahun", "score", "rating",
              "latitude", "longitude", "lat", "lon", "longitude", "percent",
              "percentage", "share", "flag", "is_"]
    return any(token in s for token in tokens)

ukuran_keywords = [
    "volume", "quantity", "tonne", "ton", "production", "produksi",
    "emission", "deforestation", "area", "distance", "cost", "flow",
    "trade", "export", "import", "amount", "weight", "ha"
]
kolom_ukuran = [
    c for c in numeric_cols
    if not nama_mirip_identitas_atau_kategori(c)
    and any(k in c.lower() for k in ukuran_keywords)
]

# Fallback: bila nama kolom tidak memakai kata kunci ukuran yang umum,
# tampilkan kandidat numerik tetapi jangan memilih kolom secara diam-diam.
print("Kolom numerik:", numeric_cols)
print("Kolom ukuran yang akan dipertimbangkan untuk imputasi median:", kolom_ukuran)

df_missing_handled = df
# Isi median hanya untuk kolom ukuran yang benar-benar memiliki null dan median non-null.
kolom_diisi_median = []
for c in kolom_ukuran:
    median_value = df_missing_handled.select(pl.col(c).median()).item()
    null_count = df_missing_handled.select(pl.col(c).null_count()).item()
    if null_count > 0 and median_value is not None:
        df_missing_handled = df_missing_handled.with_columns(
            pl.col(c).fill_null(median_value).alias(c)
        )
        kolom_diisi_median.append(c)

# Untuk kolom teks, label ini mempertahankan baris tetapi menandai informasi yang hilang.
kolom_teks = [
    name for name, dtype in zip(df_missing_handled.columns, df_missing_handled.dtypes)
    if dtype == pl.String
]
kolom_diisi_label = [
    c for c in kolom_teks
    if df_missing_handled.select(pl.col(c).null_count()).item() > 0
]
if kolom_diisi_label:
    df_missing_handled = df_missing_handled.with_columns([
        pl.col(c).fill_null("Tidak diketahui").alias(c)
        for c in kolom_diisi_label
    ])

print("Kolom numerik diisi median:", kolom_diisi_median)
print("Kolom teks diisi label:", kolom_diisi_label)
print(f"Null sebelum penanganan: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah penanganan: {df_missing_handled.null_count().sum_horizontal().sum():,}")
ringkasan_null(df_missing_handled)


## 4. Deteksi dan Penanganan Outlier

Outlier dideteksi menggunakan metode IQR pada **satu kolom ukuran yang paling relevan**. Prioritas diberikan pada nama kolom yang mengandung volume, kuantitas, produksi, emisi, luas, ekspor, impor, atau perdagangan. Tahun, kode, ID, skor, koordinat, persentase, dan indikator biner tidak dipilih secara otomatis.

Jika tidak ada kolom ukuran yang cocok, bagian ini akan melewati winsorization dan menampilkan pesan. Batas IQR adalah penanda statistik, bukan bukti bahwa nilai tersebut salah. Karena itu, nilai ekstrem hanya di-*clip* pada kolom ukuran yang terpilih, dan jumlah baris tidak dihapus.


In [ ]:
# Pilih satu kolom ukuran numerik untuk contoh penanganan outlier.
KOLOM_OUTLIER = kolom_ukuran[0] if kolom_ukuran else None
df_outlier_handled = df_missing_handled
outlier_info = {}

if KOLOM_OUTLIER is None:
    print("Tidak ditemukan kolom numerik ukuran yang aman dipilih otomatis.")
    print("Tidak ada nilai yang diubah pada tahap outlier. Tinjau daftar kolom numerik di atas.")
else:
    values = df_missing_handled.select(pl.col(KOLOM_OUTLIER).drop_nulls()).to_series()
    if len(values) == 0:
        print(f"Kolom {KOLOM_OUTLIER} tidak memiliki nilai non-null; tahap outlier dilewati.")
        KOLOM_OUTLIER = None
    else:
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        batas_bawah = q1 - 1.5 * iqr
        batas_atas = q3 + 1.5 * iqr
        jumlah_outlier = df_missing_handled.filter(
            pl.col(KOLOM_OUTLIER).is_not_null() &
            ((pl.col(KOLOM_OUTLIER) < batas_bawah) |
             (pl.col(KOLOM_OUTLIER) > batas_atas))
        ).height

        print(f"Kolom diperiksa: {KOLOM_OUTLIER}")
        print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
        print(f"Batas bawah={batas_bawah}; batas atas={batas_atas}")
        print(f"Baris di luar batas IQR: {jumlah_outlier:,}")

        # Winsorization: membatasi nilai ekstrem, bukan menghapus baris.
        df_outlier_handled = df_missing_handled.with_columns(
            pl.col(KOLOM_OUTLIER)
              .clip(lower_bound=batas_bawah, upper_bound=batas_atas)
              .alias(KOLOM_OUTLIER)
        )
        outlier_info = {
            "kolom": KOLOM_OUTLIER,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "batas_bawah": batas_bawah,
            "batas_atas": batas_atas,
            "jumlah_outlier": jumlah_outlier,
        }


## 5. Deduplikasi

Dataset rantai pasok dapat memuat banyak entitas dan dimensi; belum ada satu kolom ID unik yang dikonfirmasi dari metadata yang tersedia. Karena itu, notebook ini **menghapus baris yang identik pada seluruh kolom saja**, bukan menghapus baris hanya berdasarkan tahun atau nama perusahaan/pabrik. Jika dataset menyediakan kunci bisnis resmi, aturan deduplikasi dapat disesuaikan setelah kunci tersebut diverifikasi.


In [ ]:
baris_sebelum_deduplikasi = df_outlier_handled.height
df_dedup = df_outlier_handled.unique(maintain_order=True)
baris_sesudah_deduplikasi = df_dedup.height
jumlah_duplikat_dihapus = baris_sebelum_deduplikasi - baris_sesudah_deduplikasi

print(f"Baris sebelum deduplikasi: {baris_sebelum_deduplikasi:,}")
print(f"Baris setelah deduplikasi: {baris_sesudah_deduplikasi:,}")
print(f"Baris duplikat identik yang dihapus: {jumlah_duplikat_dihapus:,}")


## 6. Validasi Data Dasar dengan DuckDB

Validasi meliputi jumlah baris, jumlah baris duplikat yang tersisa, null per kolom, serta pemeriksaan rentang tahun bila ada kolom tahun yang dikenali. Untuk nilai kuantitatif, rentang IQR digunakan sebagai informasi diagnostik setelah penanganan; validasi tidak mengasumsikan semua nilai di luar IQR merupakan kesalahan bisnis.


In [ ]:
con.register("dataset_bersih", df_dedup)

validasi_umum = con.execute("""
    SELECT COUNT(*) AS total_baris
    FROM dataset_bersih
""").pl()
display(validasi_umum)
print(f"Duplikat identik yang tersisa setelah deduplikasi: {df_dedup.height - df_dedup.unique().height:,}")

# Ringkasan null akhir
null_summary_akhir = ringkasan_null(df_dedup)
print("Kolom yang masih memiliki null setelah cleaning:")
display(null_summary_akhir)

# Validasi tahun hanya jika ditemukan kolom tahun dengan nama yang masuk akal.
kolom_tahun = next(
    (c for c in df_dedup.columns if c.lower() in ["year", "tahun", "trade_year"]),
    None
)
if kolom_tahun:
    validasi_tahun = con.execute(
        f'SELECT MIN("{kolom_tahun}") AS tahun_min, MAX("{kolom_tahun}") AS tahun_max, '
        f'SUM(CASE WHEN "{kolom_tahun}" IS NULL THEN 1 ELSE 0 END) AS tahun_null '
        f'FROM dataset_bersih'
    ).pl()
    display(validasi_tahun)
else:
    print("Kolom tahun tidak terdeteksi dengan nama umum; validasi rentang tahun dilewati.")


## 7. Simpan Data Bersih

Hasil disimpan dalam format Parquet agar lebih efisien untuk analisis lanjutan Milestone 3. File CSV asli tetap dipertahankan di `data/raw/`.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH, compression="zstd")

# Simpan catatan ringkas agar keputusan cleaning terdokumentasi.
catatan = [
    "MILESTONE 2 — RINGKASAN DATA CLEANING",
    f"File input: {DATA_PATH.name}",
    f"Dimensi awal: {df.height:,} baris x {df.width} kolom",
    f"Dimensi akhir: {df_dedup.height:,} baris x {df_dedup.width} kolom",
    f"Kolom teks diisi label: {kolom_diisi_label}",
    f"Kolom numerik diisi median: {kolom_diisi_median}",
    f"Kolom outlier: {KOLOM_OUTLIER}",
    f"Duplikat identik dihapus: {jumlah_duplikat_dihapus:,}",
    f"Null tersisa: {df_dedup.null_count().sum_horizontal().sum():,}",
    f"Output: {OUTPUT_PATH}",
]
REPORT_PATH.write_text("\n".join(catatan), encoding="utf-8")

print(f"Baris sebelum cleaning: {df.height:,}")
print(f"Baris sesudah cleaning: {df_dedup.height:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
print(f"Catatan cleaning tersimpan di: {REPORT_PATH}")


## Catatan Milestone 2 — isi berdasarkan output aktual

Setelah seluruh sel berhasil dijalankan, tulis kesimpulan berdasarkan hasil yang benar-benar muncul:
- **Kualitas data:** kolom dengan jumlah/persentase null tertinggi dari `null_summary_awal`.
- **Missing values:** sebutkan kolom yang diisi median dan kolom teks yang diberi label; null pada kolom lain tetap dipertahankan.
- **Outlier:** tulis nama kolom yang diperiksa, batas IQR, dan jumlah nilai di luar batas. Jika tidak ada kolom yang dipilih otomatis, jelaskan bahwa tahap ini dilewati untuk menghindari perubahan kolom yang tidak sesuai.
- **Duplikasi:** tulis jumlah baris identik yang dihapus.
- **Validasi dan output:** catat hasil validasi DuckDB dan lokasi file Parquet.

Jangan menuliskan jumlah baris, nilai null, atau outlier sebelum notebook dijalankan karena angkanya bergantung pada file CSV yang ada di komputer.
